In [1]:
import duckdb                                                                                                                                  
import os
from dotenv import load_dotenv                                                                                                                 
                                                    
load_dotenv()                                                                                                                                  
con = duckdb.connect("md:warehouse_dev")

In [2]:
con.execute("""
SELECT
    table_schema,
    table_name
FROM information_schema.tables
WHERE table_type = 'BASE TABLE'
  AND table_schema IN ('bronze', 'silver')
ORDER BY table_schema, table_name;
""").df()

,table_schema,table_name
0,bronze,generation_and_exchange
1,silver,gross_consumption


In [3]:
gen_ex_df = con.execute("SELECT * FROM bronze.generation_and_exchange").df().sort_values("TimeUTC", ascending=False)
gen_ex_df.shape

(75676, 7)

In [4]:
gen_ex_df.head()

,TimeUTC,TimeDK,PriceArea,Version,GrossConsumptionMWh,CO2PerkWh,_loaded_at
75675,2026-04-26 14:00:00+02:00,2026-04-26 14:00:00,DK2,Initial,2021.403327,60.371935,2026-04-26 16:50:13.604420+02:00
75674,2026-04-26 14:00:00+02:00,2026-04-26 14:00:00,DK1,Initial,3654.471446,9.587503,2026-04-26 16:50:13.604420+02:00
75653,2026-04-26 13:00:00+02:00,2026-04-26 13:00:00,DK1,Initial,3667.213556,10.342447,2026-04-26 15:56:43.434341+02:00
75667,2026-04-26 13:00:00+02:00,2026-04-26 13:00:00,DK2,Initial,2024.332994,55.984004,2026-04-26 15:56:43.434341+02:00
75671,2026-04-26 12:00:00+02:00,2026-04-26 12:00:00,DK1,Initial,3538.073308,8.935702,2026-04-26 15:56:43.434341+02:00


In [5]:
cons_df = con.execute("SELECT * FROM silver.gross_consumption").df().sort_values("TimeUTC", ascending=False)
cons_df.shape

(75676, 5)

In [6]:
cons_df.head()

,TimeUTC,TimeDK,PriceArea,Version,GrossConsumptionMWh
75675,2026-04-26 14:00:00+02:00,2026-04-26 14:00:00,DK2,Initial,2021.403327
75674,2026-04-26 14:00:00+02:00,2026-04-26 14:00:00,DK1,Initial,3654.471446
75673,2026-04-26 13:00:00+02:00,2026-04-26 13:00:00,DK2,Initial,2024.332994
75672,2026-04-26 13:00:00+02:00,2026-04-26 13:00:00,DK1,Initial,3667.213556
75671,2026-04-26 12:00:00+02:00,2026-04-26 12:00:00,DK2,Initial,2050.858949


In [7]:
cons_df.Version.unique()

<StringArray>
['Initial', 'Preliminary', 'Final']
Length: 3, dtype: str

In [8]:
import plotly.express as px

px.line(cons_df, x='TimeDK', y='GrossConsumptionMWh', color='PriceArea', title='Gross Consumption MWh over Time')